[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wusche1/Illiad_ML_Engineering/blob/main/lectures/01_a_ml_foundations/exercises/03_architectures/notebook.ipynb)

In [ ]:
import os, importlib
if os.getenv('COLAB_RELEASE_TAG'):
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/wusche1/Illiad_ML_Engineering/main/lectures/01_a_ml_foundations/exercises/03_architectures/utils.py",
        "utils.py"
    )
    importlib.invalidate_caches()

import utils
importlib.reload(utils)
from utils import test_residual_block, test_tiny_cnn

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader

# Implementing Small Architectures

## Part A: Residual Block

Deep networks suffer from vanishing gradients: the gradient signal shrinks as it passes through many layers, making early layers hard to train. The fix is elegantly simple: add a **skip connection** that lets the input bypass the learned transformation.

$$\text{output} = F(x) + x$$

where $F$ is a small subnetwork (here: Linear $\to$ ReLU $\to$ Linear). The network only needs to learn the *residual* $F(x) = \text{output} - x$, which is easier when the desired mapping is close to identity.

**Implement `forward()`** for the `ResidualBlock` below. The `__init__` defines two linear layers and a ReLU.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, dim=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim),
            nn.ReLU(),
            nn.Linear(dim, dim),
        )

    def forward(self, x):
        # TODO: return the output of self.net(x) plus the skip connection (1 line)
        pass

In [ ]:
test_residual_block(ResidualBlock)

<details>
<summary><b>Hint</b></summary>

The skip connection adds the original input to the transformed output.
</details>

<details>
<summary><b>Solution</b></summary>

```python
class ResidualBlock(nn.Module):
    def __init__(self, dim=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim),
            nn.ReLU(),
            nn.Linear(dim, dim),
        )

    def forward(self, x):
        return self.net(x) + x
```
</details>

### Why residual connections matter

Let's compare a deep plain MLP (8 layers) against a deep residual network of the same depth. Run the cell below and watch the loss curves.

In [ ]:
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=500, noise=0.2, random_state=42)
X, y = torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)
train_set = TensorDataset(X, y)


def train(model, optimizer, dataset, epochs=50):
    loader = DataLoader(dataset, batch_size=32, shuffle=True)
    losses = []
    for _ in range(epochs):
        epoch_loss = 0.0
        for xb, yb in loader:
            pred = model(xb)
            loss = F.binary_cross_entropy_with_logits(pred, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        losses.append(epoch_loss / len(loader))
    return losses


def accuracy(model, X=X, y=y):
    with torch.no_grad():
        return ((model(X) > 0).float() == y).float().mean().item()


class DeepPlainNet(nn.Module):
    def __init__(self, dim=32, depth=8):
        super().__init__()
        layers = [nn.Linear(2, dim), nn.ReLU()]
        for _ in range(depth):
            layers += [nn.Linear(dim, dim), nn.ReLU()]
        layers.append(nn.Linear(dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


class DeepResidualNet(nn.Module):
    def __init__(self, dim=32, depth=8):
        super().__init__()
        self.input_proj = nn.Linear(2, dim)
        self.blocks = nn.Sequential(*[ResidualBlock(dim) for _ in range(depth)])
        self.head = nn.Linear(dim, 1)

    def forward(self, x):
        x = F.relu(self.input_proj(x))
        x = self.blocks(x)
        return self.head(x).squeeze(-1)


results = {}
for name, ModelClass in [('Plain (8 layers)', DeepPlainNet), ('Residual (8 layers)', DeepResidualNet)]:
    torch.manual_seed(0)
    model = ModelClass()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    losses = train(model, opt, train_set, epochs=50)
    acc = accuracy(model)
    results[name] = {'losses': losses, 'acc': acc}
    print(f"{name:25s}  accuracy: {acc:.1%}")

plt.figure(figsize=(8, 4))
for name, r in results.items():
    plt.plot(r['losses'], label=f"{name} ({r['acc']:.0%})")
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Deep Plain vs Residual Network')
plt.tight_layout()
plt.show()

---

## Part B: Tiny CNN

Architecture encodes **symmetry**. A CNN exploits translation invariance: a vertical edge is a vertical edge regardless of where it appears in the image. Instead of learning separate detectors for every position, a CNN slides the same small filter across the image.

Below is a tiny task: classify 8$\times$8 images as containing **horizontal** or **vertical** bars. The data generator is provided. **Implement `forward()`** for the CNN.

In [ ]:
def make_bars(n=500):
    images, labels = [], []
    for _ in range(n):
        img = torch.zeros(1, 8, 8)
        if torch.rand(1) > 0.5:
            row = torch.randint(0, 8, (1,))
            img[0, row, :] = 1.0
            labels.append(0)  # horizontal
        else:
            col = torch.randint(0, 8, (1,))
            img[0, :, col] = 1.0
            labels.append(1)  # vertical
        img += torch.randn_like(img) * 0.1
        images.append(img)
    return torch.stack(images), torch.tensor(labels)

x_bars, y_bars = make_bars(500)

fig, axes = plt.subplots(1, 6, figsize=(12, 2))
for i, ax in enumerate(axes):
    ax.imshow(x_bars[i, 0], cmap='gray')
    ax.set_title('H' if y_bars[i] == 0 else 'V')
    ax.axis('off')
plt.suptitle('Sample bar images')
plt.tight_layout()
plt.show()

In [ ]:
class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)   # 8x8 -> 8x8
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)  # 8x8 -> 8x8
        self.pool = nn.AdaptiveAvgPool2d(1)                       # 8x8 -> 1x1
        self.fc = nn.Linear(16, 2)

    def forward(self, x):
        # TODO: pass x through conv1 -> ReLU -> conv2 -> ReLU -> pool -> flatten -> fc
        # Pool output is (batch, 16, 1, 1) — flatten with .squeeze(-1).squeeze(-1)
        pass

In [ ]:
test_tiny_cnn(TinyCNN)

<details>
<summary><b>Hint</b></summary>

Chain the layers sequentially. After pooling, the shape is `(batch, 16, 1, 1)`. You need to remove the spatial dimensions before the linear layer.
</details>

<details>
<summary><b>Solution</b></summary>

```python
class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(16, 2)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool(x).squeeze(-1).squeeze(-1)
        return self.fc(x)
```
</details>

### Symmetry pays off: CNN vs MLP

The CNN shares weights across spatial positions. Let's see how that affects parameter count and accuracy on the bar task.

In [ ]:
class BarMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 2),
        )

    def forward(self, x):
        return self.net(x)


bar_set = TensorDataset(x_bars, y_bars)

for name, ModelClass in [('CNN', TinyCNN), ('MLP', BarMLP)]:
    torch.manual_seed(0)
    model = ModelClass()
    n_params = sum(p.numel() for p in model.parameters())
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loader = DataLoader(bar_set, batch_size=32, shuffle=True)
    for _ in range(30):
        for xb, yb in loader:
            loss = F.cross_entropy(model(xb), yb)
            opt.zero_grad()
            loss.backward()
            opt.step()
    with torch.no_grad():
        acc = (model(x_bars).argmax(1) == y_bars).float().mean().item()
    print(f"{name:4s}  params: {n_params:,}  accuracy: {acc:.1%}")